In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
TRAIN_IMG = "/content/drive/MyDrive/Dataset/train/img"
TRAIN_ANN = "/content/drive/MyDrive/Dataset/train/ann"

VAL_IMG = "/content/drive/MyDrive/Dataset/valid/img"
VAL_ANN = "/content/drive/MyDrive/Dataset/valid/ann"

In [3]:
!pip install tensorflow opencv-python scikit-learn

In [4]:
import os, json, cv2
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

In [5]:
def get_all_labels(ann_dir):
    labels = []
    for file in os.listdir(ann_dir):
        if file.endswith(".json"):
            with open(os.path.join(ann_dir, file)) as f:
                data = json.load(f)
            for obj in data.get("objects", []):
                label = obj.get("label") or obj.get("name") or obj.get("classTitle")
                if label:
                    labels.append(label)
    return labels

In [6]:
all_labels = get_all_labels(TRAIN_ANN)
encoder = LabelEncoder()
encoder.fit(all_labels)

print("Classes:", encoder.classes_)

Classes: ['ambulance' 'army vehicle' 'auto rickshaw' 'bicycle' 'bus' 'car'
 'garbagevan' 'human hauler' 'minibus' 'minivan' 'motorbike' 'pickup'
 'policecar' 'rickshaw' 'scooter' 'suv' 'taxi' 'three wheelers -CNG-'
 'truck' 'van' 'wheelbarrow']


In [7]:
def data_generator(img_dir, ann_dir, batch_size=4):
    ann_files = os.listdir(ann_dir)

    while True:
        np.random.shuffle(ann_files)
        images, labels = [], []

        for ann_file in ann_files:
            if not ann_file.endswith(".json"):
                continue

            with open(os.path.join(ann_dir, ann_file)) as f:
                data = json.load(f)

            img_path = os.path.join(img_dir, ann_file.replace(".json", ""))
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (96, 96))        # 🔽 SMALL SIZE
            img = img.astype("float32") / 255.0

            for obj in data.get("objects", []):
                label = obj.get("label") or obj.get("name") or obj.get("classTitle")
                if label:
                    images.append(img)
                    labels.append(encoder.transform([label])[0])

            if len(images) >= batch_size:
                yield np.array(images), np.array(labels)
                images, labels = [], []

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(96,96,3)),
    MaxPooling2D(),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(),

    Flatten(),
    Dense(32, activation='relu'),
    Dense(len(encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 15488)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       495,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 21)             │           693 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 501,429 (1.91 MB)

 Trainable params: 501,429 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
train_gen = data_generator(TRAIN_IMG, TRAIN_ANN, batch_size=4)
val_gen = data_generator(VAL_IMG, VAL_ANN, batch_size=4)

model.fit(
    train_gen,
    steps_per_epoch=50,   # small number = safe
    validation_data=val_gen,
    validation_steps=10,
    epochs=3
)

Epoch 1/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 971ms/step - accuracy: 0.1547 - loss: 2.9815 - val_accuracy: 0.3627 - val_loss: 2.4266
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 24s 498ms/step - accuracy: 0.1296 - loss: 2.7546 - val_accuracy: 0.1649 - val_loss: 2.3677
Epoch 3/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 430ms/step - accuracy: 0.2383 - loss: 2.5591 - val_accuracy: 0.3163 - val_loss: 2.2647


In [10]:
model.save("/content/drive/MyDrive/traffic_model.keras")
print("✅ Model saved successfully")

✅ Model saved successfully


In [11]:
import pickle

with open("/content/drive/MyDrive/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("✅ Label encoder saved")

✅ Label encoder saved
